# HW2: LLM Inference Optimization

**Setup:** `Runtime > Change runtime type > GPU` (T4 free tier works, L40S for final numbers)

Run cells top to bottom. Edit `hw2_task.py` locally, re-upload and re-run to iterate.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
!pip install -q transformers==4.57.6

In [ ]:
# Clone the repo
!git clone https://github.com/armantsaturian/gpu_and_inference_hw.git 2>/dev/null || echo "already cloned"
%cd gpu_and_inference_hw/hw2

In [ ]:
# Pull latest changes (if re-running after a push)
!git pull

In [ ]:
# Option: upload hw2_task.py manually instead of using git
# from google.colab import files
# files.upload()  # select hw2_task.py

In [ ]:
# Run the full benchmark (slow baseline + optimized)
!python3 hw2_task.py

In [ ]:
# View the profiler summary separately if needed
import importlib, sys

# Force reimport after file changes
for mod in ['hw2_task', 'utils']:
    if mod in sys.modules:
        del sys.modules[mod]

from utils import build_model, get_input_ids, PROFILE_STEPS
from hw2_task import optimized_loop, profile

model = build_model(torch.float16)
input_ids = get_input_ids()
profile(optimized_loop, model, input_ids, "v1_optimized_trace.json")
del model
torch.cuda.empty_cache()

In [ ]:
# List generated trace files
!ls -la results/

In [ ]:
# Download traces to view in ui.perfetto.dev
from google.colab import files
import os
for f in os.listdir("results"):
    if f.endswith(".json"):
        files.download(f"results/{f}")

In [ ]:
# Package submission zip
import zipfile, os

submission_files = [
    "hw2_task.py",
    "results/v0_slow_trace.json",
    "results/v1_optimized_trace.json",
]

with zipfile.ZipFile("hw2_submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for path in submission_files:
        if os.path.exists(path):
            zf.write(path)
            print(f"  added: {path}")
        else:
            print(f"  MISSING: {path}")

print("\nDone! Downloading...")
files.download("hw2_submission.zip")